# Overall Score Prediction (Student_Performance.csv)
This notebook trains and evaluates a model to predict **overall_score**, then saves the model for Streamlit.

Two training modes:
- **full**: uses most columns (drops obvious leakage like `final_grade`)
- **early**: uses only pre-exam features (does **not** use `math_score`, `science_score`, `english_score`)

In [2]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('Student_Performance.csv')  # update if needed

df = pd.read_csv(DATA_PATH)
df.head()

,student_id,age,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,math_score,science_score,english_score,overall_score,final_grade
0,1,14,male,public,post graduate,3.1,84.3,yes,<15 min,yes,notes,42.7,55.4,57.0,53.1,e
1,2,18,female,public,graduate,3.7,87.8,yes,>60 min,no,textbook,57.6,68.8,64.8,61.3,d
2,3,17,female,private,post graduate,7.9,65.5,no,<15 min,no,notes,84.8,95.0,79.2,89.6,b
3,4,16,other,public,high school,1.1,58.1,no,15-30 min,no,notes,44.4,27.5,54.7,41.6,e
4,5,16,female,public,high school,1.3,61.0,yes,30-60 min,yes,group study,8.9,32.7,30.0,25.4,f


In [3]:
df.shape, df.dtypes

((25000, 16),
 student_id                 int64
 age                        int64
 gender                    object
 school_type               object
 parent_education          object
 study_hours              float64
 attendance_percentage    float64
 internet_access           object
 travel_time               object
 extra_activities          object
 study_method              object
 math_score               float64
 science_score            float64
 english_score            float64
 overall_score            float64
 final_grade               object
 dtype: object)

In [4]:
TARGET = 'overall_score'

# Feature selection helpers

def select_features(df, mode='full'):
    drop = {'student_id', TARGET}
    if 'final_grade' in df.columns:
        drop.add('final_grade')
    if mode == 'early':
        for col in ['math_score', 'science_score', 'english_score']:
            if col in df.columns:
                drop.add(col)
    return [c for c in df.columns if c not in drop]

mode = 'full'  # change to 'early' if needed
feature_cols = select_features(df, mode)
X = df[feature_cols].copy()
y = df[TARGET].astype(float).copy()

feature_cols

['age',
 'gender',
 'school_type',
 'parent_education',
 'study_hours',
 'attendance_percentage',
 'internet_access',
 'travel_time',
 'extra_activities',
 'study_method',
 'math_score',
 'science_score',
 'english_score']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

((20000, 13), (5000, 13))

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols),
    ]
)

num_cols, cat_cols

(['age',
  'study_hours',
  'attendance_percentage',
  'math_score',
  'science_score',
  'english_score'],
 ['gender',
  'school_type',
  'parent_education',
  'internet_access',
  'travel_time',
  'extra_activities',
  'study_method'])

In [7]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

candidates = {
    'ridge': Ridge(alpha=1.0, random_state=42),
    'rf': RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
}

results = {}
best_name, best_mae, best_pipe = None, 1e9, None

for name, model in candidates.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results[name] = {'mae': mae, 'r2': r2}

    if mae < best_mae:
        best_name, best_mae, best_pipe = name, mae, pipe

best_name, results

('rf',
 {'ridge': {'mae': 3.358136317128499, 'r2': 0.9529782128287},
  'rf': {'mae': 2.195754899999998, 'r2': 0.9721733619259105}})

In [8]:
import joblib, json

# Save artifacts for Streamlit
joblib.dump(best_pipe, 'model.joblib')
meta = {
    'mode': mode,
    'best_model': best_name,
    'metrics': results,
    'feature_cols': feature_cols,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
}
Path('model_meta.json').write_text(json.dumps(meta, indent=2))

print('Saved model.joblib and model_meta.json')

Saved model.joblib and model_meta.json


## Next: run the Streamlit app
1. Put `app.py`, `model.joblib`, and `model_meta.json` in the same folder.
2. Install requirements.
3. Run: `streamlit run app.py`